# 05 - Final Load Prep for Tableau

Prepare Tableau-ready CSV files, summaries, KPIs, and final documentation.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

## Project Paths

Use relative project paths.

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

processed_data_dir = project_root / "data" / "processed"
docs_dir = project_root / "docs"
tableau_dir = project_root / "tableau"
tableau_screenshots_dir = tableau_dir / "screenshots"

processed_data_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
tableau_dir.mkdir(parents=True, exist_ok=True)
tableau_screenshots_dir.mkdir(parents=True, exist_ok=True)

cleaned_trip_data_path = processed_data_dir / "cleaned_yellow_taxi_jan_2024.parquet"

print(f"Project root: {project_root}")
print(f"Cleaned dataset path exists: {cleaned_trip_data_path.exists()}")

Project root: /Users/ishasingh/NYCTaxiTripAnalytics
Cleaned dataset path exists: True


## Load Cleaned Dataset

Load the cleaned trip-level file from Notebook 02.

In [3]:
taxi_trips = pd.read_parquet(cleaned_trip_data_path)

print(f"Cleaned dataset loaded successfully.")
print(f"Rows: {taxi_trips.shape[0]:,}")
print(f"Columns: {taxi_trips.shape[1]:,}")

taxi_trips.head()

Cleaned dataset loaded successfully.
Rows: 2,868,035
Columns: 47


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,trip_duration_minutes,passenger_count_clean,passenger_count_group,vendor_name,rate_code_label,payment_type_label,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,pickup_day_name,pickup_day_of_week,is_weekend,revenue_per_mile,fare_per_minute,tip_percentage,distance_bucket,duration_bucket,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone,is_high_value_trip,is_long_distance_trip,is_long_duration_trip
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0000,1.7200,1.0000,N,186,79,2,17.7000,1.0000,0.5000,0.0000,0.0000,1.0000,22.7000,2.5000,0.0000,19.8000,1,Solo,VeriFone Inc.,Standard rate,Cash,2024-01-01,2024,1,1,0,Monday,0,False,13.1977,0.8939,0.0000,1-3 miles,10-20 min,Manhattan,Penn Station/Madison Sq West,Yellow Zone,Manhattan,East Village,Yellow Zone,False,False,False
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0000,1.8000,1.0000,N,140,236,1,10.0000,3.5000,0.5000,3.7500,0.0000,1.0000,18.7500,2.5000,0.0000,6.6000,1,Solo,Creative Mobile Technologies,Standard rate,Credit card,2024-01-01,2024,1,1,0,Monday,0,False,10.4167,1.5152,37.5000,1-3 miles,5-10 min,Manhattan,Lenox Hill East,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone,False,False,False
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0000,4.7000,1.0000,N,236,79,1,23.3000,3.5000,0.5000,3.0000,0.0000,1.0000,31.3000,2.5000,0.0000,17.9167,1,Solo,Creative Mobile Technologies,Standard rate,Credit card,2024-01-01,2024,1,1,0,Monday,0,False,6.6596,1.3005,12.8755,3-5 miles,10-20 min,Manhattan,Upper East Side North,Yellow Zone,Manhattan,East Village,Yellow Zone,False,False,False
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0000,1.4000,1.0000,N,79,211,1,10.0000,3.5000,0.5000,2.0000,0.0000,1.0000,17.0000,2.5000,0.0000,8.3000,1,Solo,Creative Mobile Technologies,Standard rate,Credit card,2024-01-01,2024,1,1,0,Monday,0,False,12.1429,1.2048,20.0000,1-3 miles,5-10 min,Manhattan,East Village,Yellow Zone,Manhattan,SoHo,Yellow Zone,False,False,False
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0000,0.8000,1.0000,N,211,148,1,7.9000,3.5000,0.5000,3.2000,0.0000,1.0000,16.1000,2.5000,0.0000,6.1000,1,Solo,Creative Mobile Technologies,Standard rate,Credit card,2024-01-01,2024,1,1,0,Monday,0,False,20.1250,1.2951,40.5063,0-1 miles,5-10 min,Manhattan,SoHo,Yellow Zone,Manhattan,Lower East Side,Yellow Zone,False,False,False


## Final Dataset Validation

Check required Tableau fields before export.

In [4]:
required_columns = [
    "pickup_datetime",
    "pickup_date",
    "pickup_hour",
    "pickup_day_name",
    "is_weekend",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "revenue_per_mile",
    "fare_per_minute",
    "tip_percentage",
    "payment_type_label",
    "passenger_count_group",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone",
    "distance_bucket",
    "duration_bucket",
]

missing_required_columns = [
    column_name for column_name in required_columns
    if column_name not in taxi_trips.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

if len(taxi_trips) < 5_000:
    raise ValueError("The final cleaned dataset has fewer than 5,000 rows.")

print("Final dataset validation passed.")
print(f"Available rows: {len(taxi_trips):,}")

Final dataset validation passed.
Available rows: 2,868,035


## Tableau Detail Dataset

Select useful row-level fields for Tableau.

In [5]:
tableau_detail_columns = [
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_date",
    "pickup_year",
    "pickup_month",
    "pickup_day",
    "pickup_hour",
    "pickup_day_name",
    "pickup_day_of_week",
    "is_weekend",
    "vendor_name",
    "rate_code_label",
    "payment_type_label",
    "passenger_count_clean",
    "passenger_count_group",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "revenue_per_mile",
    "fare_per_minute",
    "tip_percentage",
    "distance_bucket",
    "duration_bucket",
    "pickup_location_id",
    "pickup_borough",
    "pickup_zone",
    "pickup_service_zone",
    "dropoff_location_id",
    "dropoff_borough",
    "dropoff_zone",
    "dropoff_service_zone",
    "is_high_value_trip",
    "is_long_distance_trip",
    "is_long_duration_trip",
]

tableau_detail_dataset = taxi_trips[tableau_detail_columns].copy()

# Convert date field into datetime-friendly format for CSV export.
tableau_detail_dataset["pickup_date"] = pd.to_datetime(tableau_detail_dataset["pickup_date"])

# Replace infinite values with nulls.
tableau_detail_dataset = tableau_detail_dataset.replace([np.inf, -np.inf], np.nan)

print(f"Tableau detail dataset shape: {tableau_detail_dataset.shape[0]:,} rows and {tableau_detail_dataset.shape[1]:,} columns")
tableau_detail_dataset.head()

Tableau detail dataset shape: 2,868,035 rows and 39 columns


,pickup_datetime,dropoff_datetime,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,pickup_day_name,pickup_day_of_week,is_weekend,vendor_name,rate_code_label,payment_type_label,passenger_count_clean,passenger_count_group,trip_distance,trip_duration_minutes,fare_amount,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,revenue_per_mile,fare_per_minute,tip_percentage,distance_bucket,duration_bucket,pickup_location_id,pickup_borough,pickup_zone,pickup_service_zone,dropoff_location_id,dropoff_borough,dropoff_zone,dropoff_service_zone,is_high_value_trip,is_long_distance_trip,is_long_duration_trip
0,2024-01-01 00:57:55,2024-01-01 01:17:43,2024-01-01,2024,1,1,0,Monday,0,False,VeriFone Inc.,Standard rate,Cash,1,Solo,1.7200,19.8000,17.7000,0.0000,0.0000,22.7000,2.5000,0.0000,13.1977,0.8939,0.0000,1-3 miles,10-20 min,186,Manhattan,Penn Station/Madison Sq West,Yellow Zone,79,Manhattan,East Village,Yellow Zone,False,False,False
1,2024-01-01 00:03:00,2024-01-01 00:09:36,2024-01-01,2024,1,1,0,Monday,0,False,Creative Mobile Technologies,Standard rate,Credit card,1,Solo,1.8000,6.6000,10.0000,3.7500,0.0000,18.7500,2.5000,0.0000,10.4167,1.5152,37.5000,1-3 miles,5-10 min,140,Manhattan,Lenox Hill East,Yellow Zone,236,Manhattan,Upper East Side North,Yellow Zone,False,False,False
2,2024-01-01 00:17:06,2024-01-01 00:35:01,2024-01-01,2024,1,1,0,Monday,0,False,Creative Mobile Technologies,Standard rate,Credit card,1,Solo,4.7000,17.9167,23.3000,3.0000,0.0000,31.3000,2.5000,0.0000,6.6596,1.3005,12.8755,3-5 miles,10-20 min,236,Manhattan,Upper East Side North,Yellow Zone,79,Manhattan,East Village,Yellow Zone,False,False,False
3,2024-01-01 00:36:38,2024-01-01 00:44:56,2024-01-01,2024,1,1,0,Monday,0,False,Creative Mobile Technologies,Standard rate,Credit card,1,Solo,1.4000,8.3000,10.0000,2.0000,0.0000,17.0000,2.5000,0.0000,12.1429,1.2048,20.0000,1-3 miles,5-10 min,79,Manhattan,East Village,Yellow Zone,211,Manhattan,SoHo,Yellow Zone,False,False,False
4,2024-01-01 00:46:51,2024-01-01 00:52:57,2024-01-01,2024,1,1,0,Monday,0,False,Creative Mobile Technologies,Standard rate,Credit card,1,Solo,0.8000,6.1000,7.9000,3.2000,0.0000,16.1000,2.5000,0.0000,20.1250,1.2951,40.5063,0-1 miles,5-10 min,211,Manhattan,SoHo,Yellow Zone,148,Manhattan,Lower East Side,Yellow Zone,False,False,False


## Tableau Sample Dataset

Create a reproducible sample for Tableau Public.

In [6]:
tableau_sample_size = min(300_000, len(tableau_detail_dataset))

tableau_sample_dataset = tableau_detail_dataset.sample(
    n=tableau_sample_size,
    random_state=42
).copy()

print(f"Tableau sample dataset rows: {len(tableau_sample_dataset):,}")
tableau_sample_dataset.head()

Tableau sample dataset rows: 300,000


,pickup_datetime,dropoff_datetime,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,pickup_day_name,pickup_day_of_week,is_weekend,vendor_name,rate_code_label,payment_type_label,passenger_count_clean,passenger_count_group,trip_distance,trip_duration_minutes,fare_amount,tip_amount,tolls_amount,total_amount,congestion_surcharge,airport_fee,revenue_per_mile,fare_per_minute,tip_percentage,distance_bucket,duration_bucket,pickup_location_id,pickup_borough,pickup_zone,pickup_service_zone,dropoff_location_id,dropoff_borough,dropoff_zone,dropoff_service_zone,is_high_value_trip,is_long_distance_trip,is_long_duration_trip
1286211,2024-01-16 11:01:58,2024-01-16 11:07:56,2024-01-16,2024,1,16,11,Tuesday,1,False,VeriFone Inc.,Standard rate,Credit card,2,Small group,1.0100,5.9667,7.9000,2.0000,0.0000,13.9000,2.5000,0.0000,13.7624,1.3240,25.3165,1-3 miles,5-10 min,262,Manhattan,Yorkville East,Yellow Zone,75,Manhattan,East Harlem South,Boro Zone,False,False,False
1218308,2024-01-15 12:38:25,2024-01-15 12:43:46,2024-01-15,2024,1,15,12,Monday,0,False,VeriFone Inc.,Standard rate,Credit card,1,Solo,0.9400,5.3500,7.2000,2.2400,0.0000,13.4400,2.5000,0.0000,14.2979,1.3458,31.1111,0-1 miles,5-10 min,142,Manhattan,Lincoln Square East,Yellow Zone,163,Manhattan,Midtown North,Yellow Zone,False,False,False
63533,2024-01-01 21:02:34,2024-01-01 21:10:54,2024-01-01,2024,1,1,21,Monday,0,False,VeriFone Inc.,Standard rate,Cash,1,Solo,1.3100,8.3333,10.0000,0.0000,0.0000,15.0000,2.5000,0.0000,11.4504,1.2000,0.0000,1-3 miles,5-10 min,230,Manhattan,Times Sq/Theatre District,Yellow Zone,234,Manhattan,Union Sq,Yellow Zone,False,False,False
1882591,2024-01-22 19:22:07,2024-01-22 19:27:03,2024-01-22,2024,1,22,19,Monday,0,False,VeriFone Inc.,Standard rate,Cash,1,Solo,0.7100,4.9333,6.5000,0.0000,0.0000,13.0000,2.5000,0.0000,18.3099,1.3176,0.0000,0-1 miles,0-5 min,170,Manhattan,Murray Hill,Yellow Zone,233,Manhattan,UN/Turtle Bay South,Yellow Zone,False,False,False
43247,2024-01-01 15:49:14,2024-01-01 15:56:13,2024-01-01,2024,1,1,15,Monday,0,False,VeriFone Inc.,Standard rate,Credit card,1,Solo,1.2500,6.9833,8.6000,2.5200,0.0000,15.1200,2.5000,0.0000,12.0960,1.2315,29.3023,1-3 miles,5-10 min,161,Manhattan,Midtown Center,Yellow Zone,141,Manhattan,Lenox Hill West,Yellow Zone,False,False,False


## Executive KPI Summary

Create KPI values for dashboard cards.

In [7]:
total_trips = len(tableau_detail_dataset)
total_revenue = tableau_detail_dataset["total_amount"].sum()
average_total_amount = tableau_detail_dataset["total_amount"].mean()
average_fare_amount = tableau_detail_dataset["fare_amount"].mean()
average_trip_distance = tableau_detail_dataset["trip_distance"].mean()
average_trip_duration = tableau_detail_dataset["trip_duration_minutes"].mean()
average_revenue_per_mile = tableau_detail_dataset["revenue_per_mile"].mean()
average_tip_amount = tableau_detail_dataset["tip_amount"].mean()
average_tip_percentage = tableau_detail_dataset["tip_percentage"].mean()
credit_card_payment_share = tableau_detail_dataset["payment_type_label"].eq("Credit card").mean() * 100

peak_hour = (
    tableau_detail_dataset
    .groupby("pickup_hour")
    .size()
    .sort_values(ascending=False)
    .index[0]
)

top_pickup_borough = (
    tableau_detail_dataset
    .groupby("pickup_borough")
    .size()
    .sort_values(ascending=False)
    .index[0]
)

top_dropoff_borough = (
    tableau_detail_dataset
    .groupby("dropoff_borough")
    .size()
    .sort_values(ascending=False)
    .index[0]
)

executive_kpi_summary = pd.DataFrame({
    "kpi_name": [
        "Total Trips",
        "Total Revenue",
        "Average Total Amount",
        "Average Fare Amount",
        "Average Trip Distance",
        "Average Trip Duration",
        "Average Revenue per Mile",
        "Average Tip Amount",
        "Average Tip Percentage",
        "Credit Card Payment Share",
        "Peak Pickup Hour",
        "Top Pickup Borough",
        "Top Drop-off Borough",
    ],
    "kpi_value": [
        total_trips,
        total_revenue,
        average_total_amount,
        average_fare_amount,
        average_trip_distance,
        average_trip_duration,
        average_revenue_per_mile,
        average_tip_amount,
        average_tip_percentage,
        credit_card_payment_share,
        peak_hour,
        top_pickup_borough,
        top_dropoff_borough,
    ],
    "kpi_unit": [
        "Trips",
        "USD",
        "USD per trip",
        "USD per trip",
        "Miles",
        "Minutes",
        "USD per mile",
        "USD",
        "Percentage",
        "Percentage",
        "Hour",
        "Borough",
        "Borough",
    ],
})

executive_kpi_summary

,kpi_name,kpi_value,kpi_unit
0,Total Trips,2868035,Trips
1,Total Revenue,"78,407,817.4100",USD
2,Average Total Amount,27.3385,USD per trip
3,Average Fare Amount,18.4894,USD per trip
4,Average Trip Distance,3.7318,Miles
5,Average Trip Duration,14.9565,Minutes
6,Average Revenue per Mile,17.4880,USD per mile
7,Average Tip Amount,3.4025,USD
8,Average Tip Percentage,21.4156,Percentage
9,Credit Card Payment Share,80.0894,Percentage


## Daily Summary

Daily demand and revenue trends.

In [8]:
daily_summary = (
    tableau_detail_dataset
    .groupby("pickup_date")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
)

daily_summary["daily_revenue_growth_percentage"] = (
    daily_summary["total_revenue"]
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
    * 100
)

daily_summary.head()

,pickup_date,total_trips,total_revenue,average_total_amount,average_fare_amount,average_tip_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile,daily_revenue_growth_percentage
0,2024-01-01,75443,"2,335,768.4500",30.9607,22.1892,3.4300,4.6552,15.1015,18.4020,NaN
1,2024-01-02,72944,"2,251,733.6400",30.8693,21.3513,3.5879,4.2071,16.0164,17.2536,-3.5977
2,2024-01-03,79907,"2,337,062.7900",29.2473,20.0686,3.4278,3.9504,15.8194,17.2677,3.7895
3,2024-01-04,100089,"2,778,329.1700",27.7586,18.7550,3.3519,3.3586,15.3527,17.4998,18.8812
4,2024-01-05,100079,"2,699,790.8200",26.9766,18.1458,3.2875,3.8265,14.7033,16.5910,-2.8268


## Hourly Summary

Peak-hour demand and revenue.

In [9]:
hourly_summary = (
    tableau_detail_dataset
    .groupby("pickup_hour")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
)

hourly_summary["trip_share_percentage"] = (
    hourly_summary["total_trips"] / hourly_summary["total_trips"].sum() * 100
)

hourly_summary

,pickup_hour,total_trips,total_revenue,average_total_amount,average_fare_amount,average_tip_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile,trip_share_percentage
0,0,75166,"2,147,600.7300",28.5714,19.6875,3.4200,3.8531,13.7250,16.8455,2.6208
1,1,50437,"1,306,245.2800",25.8986,17.7279,3.0368,3.2668,12.6909,17.4998,1.7586
2,2,34937,"859,245.7700",24.5941,16.6200,2.8904,3.0395,12.0119,17.1422,1.2182
3,3,22923,"612,113.3000",26.7030,18.5239,2.9396,3.5152,12.1545,18.8687,0.7993
4,4,15263,"495,412.9800",32.4584,23.4258,3.3044,4.8694,13.9946,19.3047,0.5322
5,5,17487,"656,879.4300",37.5639,27.5000,3.9650,9.1800,16.5329,21.4928,0.6097
6,6,39393,"1,187,456.4100",30.1438,22.0175,3.2335,13.4764,15.7368,16.1785,1.3735
7,7,80824,"2,143,610.3800",26.5220,18.7429,3.1425,6.1782,14.9966,15.0766,2.8181
8,8,113451,"2,897,223.3200",25.5372,17.8296,3.1183,5.5941,15.1792,16.3703,3.9557
9,9,125541,"3,250,967.7900",25.8957,17.9462,3.2079,3.0497,15.1214,16.4302,4.3772


## Day-of-Week Summary

Weekday and weekend comparison.

In [10]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_summary = (
    tableau_detail_dataset
    .groupby(["pickup_day_of_week", "pickup_day_name", "is_weekend"])
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
    .sort_values("pickup_day_of_week")
)

weekday_summary["day_type"] = np.where(
    weekday_summary["is_weekend"],
    "Weekend",
    "Weekday"
)

weekday_summary

,pickup_day_of_week,pickup_day_name,is_weekend,total_trips,total_revenue,average_total_amount,average_fare_amount,average_tip_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile,day_type
0,0,Monday,False,393005,"11,331,302.8000",28.8325,19.7074,3.5320,3.7740,14.9638,17.4334,Weekday
1,1,Tuesday,False,448879,"12,404,110.5700",27.6335,18.6044,3.4348,4.2444,15.4717,17.5346,Weekday
2,2,Wednesday,False,480554,"13,220,799.1800",27.5116,18.4482,3.4516,3.6081,15.5196,17.6079,Weekday
3,3,Thursday,False,416030,"11,532,275.3600",27.7198,18.5987,3.4788,3.5391,15.7838,17.6858,Weekday
4,4,Friday,False,396156,"10,767,101.2700",27.1789,18.1879,3.3910,3.6746,15.1606,17.2470,Weekday
5,5,Saturday,True,406923,"10,254,988.9100",25.2013,17.2441,3.1319,3.3873,13.8994,17.2498,Weekend
6,6,Sunday,True,326488,"8,897,239.3200",27.2514,18.7040,3.3839,3.9026,13.4265,17.6501,Weekend


## Pickup Borough Summary

Origin borough demand, revenue, and efficiency.

In [11]:
pickup_borough_summary = (
    tableau_detail_dataset
    .groupby("pickup_borough")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_tip_percentage=("tip_percentage", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
    .sort_values("total_trips", ascending=False)
)

pickup_borough_summary["trip_share_percentage"] = (
    pickup_borough_summary["total_trips"] / pickup_borough_summary["total_trips"].sum() * 100
)

pickup_borough_summary

,pickup_borough,total_trips,total_revenue,average_total_amount,median_total_amount,average_fare_amount,average_tip_amount,average_tip_percentage,average_trip_distance,average_trip_duration,average_revenue_per_mile,trip_share_percentage
3,Manhattan,2572024,"58,534,866.1400",22.7583,19.2500,14.9179,2.9080,21.5532,2.7171,13.0119,17.0235,89.6790
4,Queens,257603,"18,603,571.2000",72.2180,72.7800,52.7756,8.5558,21.8288,12.8148,32.3984,19.2825,8.9819
1,Brooklyn,22254,"738,652.1100",33.1919,29.0000,28.7674,1.6201,6.9206,14.8153,31.6431,22.6189,0.7759
6,Unknown,10316,"319,584.3300",30.9795,20.3000,22.4587,3.6009,19.6591,3.7253,15.3524,66.2212,0.3597
0,Bronx,5742,"203,476.1300",35.4365,34.0000,32.3475,0.2222,0.7849,7.7331,38.1664,15.7615,0.2002
2,EWR,50,"4,860.0400",97.2008,108.6000,82.6870,10.3592,11.3992,5.9988,7.5893,"2,501.1979",0.0017
5,Staten Island,46,"2,807.4600",61.0317,38.4900,45.9220,4.4098,5.6705,9.9867,20.1793,41.5584,0.0016


## Drop-off Borough Summary

Destination borough patterns.

In [12]:
dropoff_borough_summary = (
    tableau_detail_dataset
    .groupby("dropoff_borough")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_tip_percentage=("tip_percentage", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
    .sort_values("total_trips", ascending=False)
)

dropoff_borough_summary["trip_share_percentage"] = (
    dropoff_borough_summary["total_trips"] / dropoff_borough_summary["total_trips"].sum() * 100
)

dropoff_borough_summary

,dropoff_borough,total_trips,total_revenue,average_total_amount,median_total_amount,average_fare_amount,average_tip_amount,average_tip_percentage,average_trip_distance,average_trip_duration,average_revenue_per_mile,trip_share_percentage
3,Manhattan,2583234,"62,203,035.4600",24.0795,19.3000,15.7395,3.0959,21.6107,3.0254,13.4167,16.7749,90.0698
4,Queens,133282,"7,361,113.5900",55.2296,53.3300,40.9619,6.1504,14.7033,10.0002,27.7390,30.7897,4.6472
1,Brooklyn,103001,"5,306,889.6900",51.5227,48.2400,40.2719,5.9173,15.2446,9.0227,30.5889,9.6757,3.5913
6,Unknown,24911,"1,695,553.5300",68.0645,35.7500,53.8531,6.8771,73.7586,9.9840,23.7790,55.7024,0.8686
0,Bronx,16197,"901,182.3600",55.6388,52.4900,45.2581,3.3012,6.8697,14.6802,32.1432,9.4774,0.5647
2,EWR,6804,"870,873.9800",127.9944,124.6200,93.3804,14.9097,16.1643,18.4395,36.8717,25.7273,0.2372
5,Staten Island,606,"69,168.8000",114.1399,115.0000,85.7280,9.3878,10.7691,22.0310,42.4284,9.9877,0.0211


## Pickup Zone Summary

High-demand pickup zones.

In [13]:
pickup_zone_summary = (
    tableau_detail_dataset
    .groupby(["pickup_borough", "pickup_zone"])
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
        average_tip_percentage=("tip_percentage", "mean"),
    )
    .reset_index()
    .sort_values("total_trips", ascending=False)
)

pickup_zone_summary["trip_share_percentage"] = (
    pickup_zone_summary["total_trips"] / pickup_zone_summary["total_trips"].sum() * 100
)

pickup_zone_summary.head(20)

,pickup_borough,pickup_zone,total_trips,total_revenue,average_total_amount,median_total_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile,average_tip_percentage,trip_share_percentage
143,Manhattan,Midtown Center,140074,"3,354,213.0500",23.9460,20.1000,2.5985,13.9043,18.0647,21.6457,4.8840
162,Manhattan,Upper East Side South,140069,"2,770,461.1500",19.7793,17.2800,1.7152,11.0270,17.2501,22.8086,4.8838
202,Queens,JFK Airport,138311,"11,183,307.4600",80.8562,87.6900,15.9390,37.1085,20.4420,14.1911,4.8225
161,Manhattan,Upper East Side North,133901,"2,712,549.6900",20.2579,17.6400,1.8707,11.2233,16.6637,22.5706,4.6687
144,Manhattan,Midtown East,104298,"2,433,071.9100",23.3281,19.8000,2.2587,13.3749,17.2196,21.9152,3.6366
156,Manhattan,Times Sq/Theatre District,102890,"2,763,318.2200",26.8570,20.3000,2.9693,15.0898,20.2584,20.6228,3.5875
149,Manhattan,Penn Station/Madison Sq West,102099,"2,461,096.7000",24.1050,20.6500,2.3030,15.1740,16.9187,20.8949,3.5599
135,Manhattan,Lincoln Square East,101706,"2,169,364.0200",21.3298,18.8100,2.1173,11.7689,15.4758,22.4487,3.5462
209,Queens,LaGuardia Airport,87658,"5,817,091.8600",66.3612,66.4900,9.7104,27.1501,12.5175,21.2548,3.0564
164,Manhattan,Upper West Side South,86432,"1,833,747.8700",21.2161,18.4800,2.2933,11.5711,15.2013,22.7639,3.0136


## Drop-off Zone Summary

Top destination zones.

In [14]:
dropoff_zone_summary = (
    tableau_detail_dataset
    .groupby(["dropoff_borough", "dropoff_zone"])
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
        average_tip_percentage=("tip_percentage", "mean"),
    )
    .reset_index()
    .sort_values("total_trips", ascending=False)
)

dropoff_zone_summary["trip_share_percentage"] = (
    dropoff_zone_summary["total_trips"] / dropoff_zone_summary["total_trips"].sum() * 100
)

dropoff_zone_summary.head(20)

,dropoff_borough,dropoff_zone,total_trips,total_revenue,average_total_amount,median_total_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile,average_tip_percentage,trip_share_percentage
160,Manhattan,Upper East Side North,139349,"2,952,274.1400",21.1862,18.0000,2.7555,11.3731,16.3033,23.0532,4.8587
161,Manhattan,Upper East Side South,127599,"2,549,331.1500",19.9792,17.1600,1.8342,11.0064,17.7359,22.9796,4.4490
142,Manhattan,Midtown Center,109213,"2,530,428.8700",23.1697,18.7100,2.8544,13.4637,19.8686,21.3637,3.8079
155,Manhattan,Times Sq/Theatre District,87580,"2,580,696.0500",29.4667,20.3000,3.3049,17.0779,22.7961,19.7952,3.0537
134,Manhattan,Lincoln Square East,87575,"1,935,931.4300",22.1060,18.8000,2.5569,12.2842,16.9810,22.7077,3.0535
163,Manhattan,Upper West Side South,87183,"2,022,243.7900",23.1954,18.9600,2.4737,12.4155,14.6967,23.2305,3.0398
147,Manhattan,Murray Hill,84493,"1,928,950.0600",22.8297,18.4800,2.2289,12.3564,17.8278,21.9626,2.9460
143,Manhattan,Midtown East,83102,"1,932,606.8200",23.2558,18.7500,7.1841,12.8486,18.1206,21.8314,2.8975
133,Manhattan,Lenox Hill West,81666,"1,784,315.5200",21.8489,18.1200,2.1675,11.6578,16.8818,22.4326,2.8475
114,Manhattan,East Chelsea,72324,"1,731,575.6000",23.9419,19.2000,2.5139,13.3625,17.2667,27.5328,2.5217


## Borough Flow Summary

Pickup-to-drop-off borough movement.

In [15]:
borough_flow_summary = (
    tableau_detail_dataset
    .groupby(["pickup_borough", "dropoff_borough"])
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
    .sort_values("total_trips", ascending=False)
)

borough_flow_summary["trip_share_percentage"] = (
    borough_flow_summary["total_trips"] / borough_flow_summary["total_trips"].sum() * 100
)

borough_flow_summary.head(20)

,pickup_borough,dropoff_borough,total_trips,total_revenue,average_total_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile,trip_share_percentage
21,Manhattan,Manhattan,2422225,"49,741,151.4700",20.5353,2.2460,11.8957,17.4196,84.4559
28,Queens,Manhattan,148155,"11,985,554.4200",80.8988,14.1333,36.4178,6.9094,5.1657
22,Manhattan,Queens,74782,"4,923,979.0100",65.8444,12.2202,34.0645,6.8141,2.6074
29,Queens,Queens,54805,"2,264,481.0600",41.3189,6.9209,18.2890,65.0821,1.9109
19,Manhattan,Brooklyn,51029,"2,236,178.3300",43.8217,6.5279,28.3622,7.4083,1.7792
26,Queens,Brooklyn,40000,"2,768,876.5500",69.2219,13.6779,35.1038,5.5303,1.3947
7,Brooklyn,Brooklyn,11209,"258,328.7200",23.0465,3.3697,22.4663,34.9606,0.3908
24,Manhattan,Unknown,9683,"468,585.7200",48.3926,5.9796,19.1094,62.0024,0.3376
31,Queens,Unknown,8085,"1,007,344.1200",124.5942,20.4837,37.7737,14.6812,0.2819
18,Manhattan,Bronx,7754,"351,495.4600",45.3309,16.3185,30.2770,7.2368,0.2704


## Zone Flow Summary

Pickup-zone to drop-off-zone flows with 50+ trips.

In [16]:
zone_flow_summary = (
    tableau_detail_dataset
    .groupby(["pickup_borough", "pickup_zone", "dropoff_borough", "dropoff_zone"])
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
)

zone_flow_summary = zone_flow_summary[
    zone_flow_summary["total_trips"] >= 50
].copy()

zone_flow_summary = zone_flow_summary.sort_values(
    "total_trips",
    ascending=False
)

zone_flow_summary.head(20)

,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,total_trips,total_revenue,average_total_amount,average_trip_distance,average_trip_duration,average_revenue_per_mile
17818,Manhattan,Upper East Side South,Manhattan,Upper East Side North,21635,"340,503.5900",15.7386,1.0614,7.3858,16.7397
17615,Manhattan,Upper East Side North,Manhattan,Upper East Side South,19189,"308,534.8800",16.0787,1.0489,8.0292,17.1029
17614,Manhattan,Upper East Side North,Manhattan,Upper East Side North,15190,"200,704.1500",13.2129,0.6481,4.7977,39.0429
17819,Manhattan,Upper East Side South,Manhattan,Upper East Side South,14107,"194,743.0400",13.8047,0.6488,5.7182,38.7257
14443,Manhattan,Midtown Center,Manhattan,Upper East Side South,10138,"172,942.4600",17.0588,1.0775,9.1689,17.2764
13235,Manhattan,Lincoln Square East,Manhattan,Upper West Side South,8820,"132,096.4300",14.9769,0.9920,6.1741,16.5560
17800,Manhattan,Upper East Side South,Manhattan,Midtown Center,8716,"143,617.2600",16.4774,1.0615,8.9607,16.8718
14442,Manhattan,Midtown Center,Manhattan,Upper East Side North,8660,"189,522.6800",21.8848,1.9533,13.2265,11.4626
18172,Manhattan,Upper West Side South,Manhattan,Lincoln Square East,8539,"124,515.2900",14.5820,0.8848,6.1096,19.7969
18200,Manhattan,Upper West Side South,Manhattan,Upper West Side North,8301,"114,851.9400",13.8359,0.8405,5.1125,19.2409


## Payment Type Summary

Payment and tipping behavior.

In [17]:
payment_type_summary = (
    tableau_detail_dataset
    .groupby("payment_type_label")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        median_tip_amount=("tip_amount", "median"),
        average_tip_percentage=("tip_percentage", "mean"),
        median_tip_percentage=("tip_percentage", "median"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
    )
    .reset_index()
    .sort_values("total_trips", ascending=False)
)

payment_type_summary["trip_share_percentage"] = (
    payment_type_summary["total_trips"] / payment_type_summary["total_trips"].sum() * 100
)

payment_type_summary

,payment_type_label,total_trips,total_revenue,average_total_amount,median_total_amount,average_fare_amount,average_tip_amount,median_tip_amount,average_tip_percentage,median_tip_percentage,average_trip_distance,average_trip_duration,trip_share_percentage
1,Credit card,2296991,"64,481,551.0500",28.0722,20.6400,18.3722,4.1558,3.1400,26.2805,26.2500,3.2930,14.9696,80.0894
0,Cash,422295,"10,064,127.3700",23.8320,16.8000,18.5900,0.0001,0.0000,0.0010,0.0000,3.3132,14.7141,14.7242
4,Unknown,115237,"3,054,982.5600",26.5104,22.0800,20.3572,1.8442,0.0000,9.1381,0.0000,14.1538,16.1040,4.0180
2,Dispute,22874,"571,757.7800",24.9960,16.5000,19.6578,0.0038,0.0000,0.0295,0.0000,3.4281,13.5963,0.7975
3,No charge,10638,"235,398.6500",22.1281,15.0000,17.0480,0.0050,0.0000,0.0394,0.0000,2.8569,12.2565,0.3709


## Distance Bucket Summary

Trip distance segments.

In [18]:
distance_bucket_summary = (
    tableau_detail_dataset
    .groupby("distance_bucket")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
        median_revenue_per_mile=("revenue_per_mile", "median"),
    )
    .reset_index()
)

distance_bucket_summary["trip_share_percentage"] = (
    distance_bucket_summary["total_trips"] / distance_bucket_summary["total_trips"].sum() * 100
)

distance_bucket_summary

,distance_bucket,total_trips,total_revenue,average_total_amount,median_total_amount,average_fare_amount,average_tip_amount,average_trip_duration,average_revenue_per_mile,median_revenue_per_mile,trip_share_percentage
0,0-1 miles,704747,"10,065,419.8000",14.2823,13.5000,7.8085,1.8275,5.9894,39.7477,19.3205,24.5725
1,1-3 miles,1402648,"28,646,906.2000",20.4234,19.7500,13.0783,2.6505,12.3004,12.0484,11.6496,48.9062
2,3-5 miles,305453,"9,437,188.5600",30.8957,30.2400,22.2470,3.8523,20.6328,8.2101,8.0481,10.6503
3,5-10 miles,230426,"10,996,596.3900",47.7229,44.6000,34.0384,5.7375,25.7090,6.5768,6.5614,8.0343
4,10-20 miles,195889,"15,965,705.3600",81.5038,82.6900,60.9286,9.6904,39.5955,5.5421,5.4366,6.8301
5,20+ miles,28872,"3,296,001.1000",114.1591,99.1900,90.2894,12.3265,49.8404,4.6945,4.5905,1.0067


## Duration Bucket Summary

Trip duration segments.

In [19]:
duration_bucket_summary = (
    tableau_detail_dataset
    .groupby("duration_bucket")
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        median_total_amount=("total_amount", "median"),
        average_fare_amount=("fare_amount", "mean"),
        average_tip_amount=("tip_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_revenue_per_mile=("revenue_per_mile", "mean"),
    )
    .reset_index()
)

duration_bucket_summary["trip_share_percentage"] = (
    duration_bucket_summary["total_trips"] / duration_bucket_summary["total_trips"].sum() * 100
)

duration_bucket_summary

,duration_bucket,total_trips,total_revenue,average_total_amount,median_total_amount,average_fare_amount,average_tip_amount,average_trip_distance,average_revenue_per_mile,trip_share_percentage
0,0-5 min,336887,"4,441,346.5200",13.1835,12.2000,6.8845,1.6799,0.8381,59.0863,11.7463
1,5-10 min,842378,"13,567,820.8700",16.1066,15.9600,9.3074,2.1020,1.3158,15.2124,29.3713
2,10-20 min,1054485,"25,442,699.8100",24.1281,22.3200,16.0645,3.0621,3.2542,11.5539,36.7668
3,20-30 min,372680,"15,437,147.2800",41.4220,34.7000,29.7785,5.1774,6.1549,9.0892,12.9943
4,30-60 min,233646,"17,053,580.4300",72.9890,75.3100,54.9975,8.6960,13.2637,7.1746,8.1466
5,60+ min,27959,"2,465,222.5000",88.1728,87.6900,70.8487,8.2859,17.4498,6.7605,0.9748


## Hour-by-Weekday Heatmap

Demand by hour and weekday.

In [20]:
hour_weekday_heatmap = (
    tableau_detail_dataset
    .groupby(["pickup_day_of_week", "pickup_day_name", "pickup_hour"])
    .agg(
        total_trips=("pickup_datetime", "count"),
        total_revenue=("total_amount", "sum"),
        average_total_amount=("total_amount", "mean"),
        average_trip_distance=("trip_distance", "mean"),
        average_trip_duration=("trip_duration_minutes", "mean"),
    )
    .reset_index()
    .sort_values(["pickup_day_of_week", "pickup_hour"])
)

hour_weekday_heatmap.head()

,pickup_day_of_week,pickup_day_name,pickup_hour,total_trips,total_revenue,average_total_amount,average_trip_distance,average_trip_duration
0,0,Monday,0,11313,"356,189.4200",31.4850,4.2671,15.4436
1,0,Monday,1,9014,"262,510.7400",29.1226,3.5679,15.4415
2,0,Monday,2,6876,"187,775.1600",27.3088,3.2664,13.9912
3,0,Monday,3,5331,"151,892.8100",28.4924,3.5294,13.3125
4,0,Monday,4,3893,"121,115.4600",31.1111,4.3994,13.8033


## Outlier Flag Summary

Counts for key trip flags.

In [21]:
outlier_flag_summary = pd.DataFrame({
    "outlier_flag": [
        "High Value Trip",
        "Long Distance Trip",
        "Long Duration Trip",
    ],
    "flagged_trip_count": [
        int(tableau_detail_dataset["is_high_value_trip"].sum()),
        int(tableau_detail_dataset["is_long_distance_trip"].sum()),
        int(tableau_detail_dataset["is_long_duration_trip"].sum()),
    ],
})

outlier_flag_summary["flagged_trip_percentage"] = (
    outlier_flag_summary["flagged_trip_count"] / len(tableau_detail_dataset) * 100
)

outlier_flag_summary

,outlier_flag,flagged_trip_count,flagged_trip_percentage
0,High Value Trip,28210,0.9836
1,Long Distance Trip,28598,0.9971
2,Long Duration Trip,28669,0.9996


## Save Tableau Files

Export final Tableau-ready CSV files.

In [22]:
output_files = {
    "tableau_taxi_trips_sample.csv": tableau_sample_dataset,
    "tableau_executive_kpi_summary.csv": executive_kpi_summary,
    "tableau_daily_summary.csv": daily_summary,
    "tableau_hourly_summary.csv": hourly_summary,
    "tableau_weekday_summary.csv": weekday_summary,
    "tableau_pickup_borough_summary.csv": pickup_borough_summary,
    "tableau_dropoff_borough_summary.csv": dropoff_borough_summary,
    "tableau_pickup_zone_summary.csv": pickup_zone_summary,
    "tableau_dropoff_zone_summary.csv": dropoff_zone_summary,
    "tableau_borough_flow_summary.csv": borough_flow_summary,
    "tableau_zone_flow_summary.csv": zone_flow_summary,
    "tableau_payment_type_summary.csv": payment_type_summary,
    "tableau_distance_bucket_summary.csv": distance_bucket_summary,
    "tableau_duration_bucket_summary.csv": duration_bucket_summary,
    "tableau_hour_weekday_heatmap.csv": hour_weekday_heatmap,
    "tableau_outlier_flag_summary.csv": outlier_flag_summary,
}

for file_name, dataframe in output_files.items():
    output_path = processed_data_dir / file_name
    dataframe.to_csv(output_path, index=False)
    print(f"Saved {file_name}: {dataframe.shape[0]:,} rows and {dataframe.shape[1]:,} columns")

Saved tableau_taxi_trips_sample.csv: 300,000 rows and 39 columns
Saved tableau_executive_kpi_summary.csv: 13 rows and 3 columns
Saved tableau_daily_summary.csv: 31 rows and 10 columns
Saved tableau_hourly_summary.csv: 24 rows and 10 columns
Saved tableau_weekday_summary.csv: 7 rows and 12 columns
Saved tableau_pickup_borough_summary.csv: 7 rows and 12 columns
Saved tableau_dropoff_borough_summary.csv: 7 rows and 12 columns
Saved tableau_pickup_zone_summary.csv: 257 rows and 11 columns
Saved tableau_dropoff_zone_summary.csv: 260 rows and 11 columns
Saved tableau_borough_flow_summary.csv: 46 rows and 9 columns
Saved tableau_zone_flow_summary.csv: 3,885 rows and 10 columns
Saved tableau_payment_type_summary.csv: 5 rows and 13 columns
Saved tableau_distance_bucket_summary.csv: 6 rows and 11 columns
Saved tableau_duration_bucket_summary.csv: 6 rows and 10 columns
Saved tableau_hour_weekday_heatmap.csv: 168 rows and 8 columns
Saved tableau_outlier_flag_summary.csv: 3 rows and 3 columns


## Instructor Preview File

Save a smaller row-level preview.

In [23]:
instructor_preview_dataset = tableau_detail_dataset.sample(
    n=min(10_000, len(tableau_detail_dataset)),
    random_state=123
).copy()

instructor_preview_path = processed_data_dir / "instructor_preview_sample_10000.csv"
instructor_preview_dataset.to_csv(instructor_preview_path, index=False)

print(f"Instructor preview file saved to: {instructor_preview_path}")
print(f"Preview rows: {len(instructor_preview_dataset):,}")

Instructor preview file saved to: /Users/ishasingh/NYCTaxiTripAnalytics/data/processed/instructor_preview_sample_10000.csv
Preview rows: 10,000


## Final Data Dictionary

Describe the fields in the main Tableau dataset.

In [24]:
data_dictionary_entries = [
    {
        "column_name": "pickup_datetime",
        "description": "Date and time when the taxi trip started.",
        "data_type": "datetime",
        "example_use": "Time trend analysis and filtering.",
    },
    {
        "column_name": "dropoff_datetime",
        "description": "Date and time when the taxi trip ended.",
        "data_type": "datetime",
        "example_use": "Trip duration calculation.",
    },
    {
        "column_name": "pickup_date",
        "description": "Date of taxi pickup.",
        "data_type": "date",
        "example_use": "Daily demand and revenue trend analysis.",
    },
    {
        "column_name": "pickup_hour",
        "description": "Hour of day when the trip started, from 0 to 23.",
        "data_type": "integer",
        "example_use": "Peak demand hour analysis.",
    },
    {
        "column_name": "pickup_day_name",
        "description": "Day name of trip pickup.",
        "data_type": "categorical",
        "example_use": "Weekday demand comparison.",
    },
    {
        "column_name": "is_weekend",
        "description": "Boolean flag identifying Saturday and Sunday trips.",
        "data_type": "boolean",
        "example_use": "Weekday vs weekend comparison.",
    },
    {
        "column_name": "vendor_name",
        "description": "Readable taxi technology vendor name.",
        "data_type": "categorical",
        "example_use": "Vendor-level trip comparison.",
    },
    {
        "column_name": "rate_code_label",
        "description": "Readable rate category such as Standard rate, JFK, Newark, or Negotiated fare.",
        "data_type": "categorical",
        "example_use": "Fare type analysis.",
    },
    {
        "column_name": "payment_type_label",
        "description": "Readable payment type such as Credit card or Cash.",
        "data_type": "categorical",
        "example_use": "Payment and tipping behavior analysis.",
    },
    {
        "column_name": "passenger_count_clean",
        "description": "Cleaned passenger count field.",
        "data_type": "integer",
        "example_use": "Passenger group analysis.",
    },
    {
        "column_name": "passenger_count_group",
        "description": "Passenger count grouped as Solo, Small group, Large group, Unknown, or Zero reported.",
        "data_type": "categorical",
        "example_use": "Segmented demand analysis.",
    },
    {
        "column_name": "trip_distance",
        "description": "Trip distance in miles.",
        "data_type": "numeric",
        "example_use": "Distance and revenue efficiency analysis.",
    },
    {
        "column_name": "trip_duration_minutes",
        "description": "Trip duration in minutes.",
        "data_type": "numeric",
        "example_use": "Duration and congestion-related analysis.",
    },
    {
        "column_name": "fare_amount",
        "description": "Base fare charged for the taxi trip.",
        "data_type": "numeric",
        "example_use": "Fare analysis.",
    },
    {
        "column_name": "tip_amount",
        "description": "Recorded tip amount for the trip.",
        "data_type": "numeric",
        "example_use": "Tip behavior analysis.",
    },
    {
        "column_name": "total_amount",
        "description": "Total amount charged to passengers including fare, tips, taxes, surcharges, tolls, and fees.",
        "data_type": "numeric",
        "example_use": "Revenue analysis.",
    },
    {
        "column_name": "revenue_per_mile",
        "description": "Total amount divided by trip distance.",
        "data_type": "numeric",
        "example_use": "Trip revenue efficiency analysis.",
    },
    {
        "column_name": "fare_per_minute",
        "description": "Fare amount divided by trip duration in minutes.",
        "data_type": "numeric",
        "example_use": "Trip time efficiency analysis.",
    },
    {
        "column_name": "tip_percentage",
        "description": "Tip amount divided by fare amount, expressed as a percentage.",
        "data_type": "numeric",
        "example_use": "Tip behavior analysis.",
    },
    {
        "column_name": "distance_bucket",
        "description": "Categorical grouping of trip distance.",
        "data_type": "categorical",
        "example_use": "Short, medium, and long trip analysis.",
    },
    {
        "column_name": "duration_bucket",
        "description": "Categorical grouping of trip duration.",
        "data_type": "categorical",
        "example_use": "Duration segment analysis.",
    },
    {
        "column_name": "pickup_borough",
        "description": "NYC borough where the trip started.",
        "data_type": "categorical",
        "example_use": "Origin demand analysis.",
    },
    {
        "column_name": "pickup_zone",
        "description": "Taxi zone where the trip started.",
        "data_type": "categorical",
        "example_use": "High-demand pickup zone analysis.",
    },
    {
        "column_name": "dropoff_borough",
        "description": "NYC borough where the trip ended.",
        "data_type": "categorical",
        "example_use": "Destination demand analysis.",
    },
    {
        "column_name": "dropoff_zone",
        "description": "Taxi zone where the trip ended.",
        "data_type": "categorical",
        "example_use": "High-demand destination zone analysis.",
    },
    {
        "column_name": "is_high_value_trip",
        "description": "Flag for trips above the 99th percentile of total amount.",
        "data_type": "boolean",
        "example_use": "Outlier and premium trip analysis.",
    },
    {
        "column_name": "is_long_distance_trip",
        "description": "Flag for trips above the 99th percentile of trip distance.",
        "data_type": "boolean",
        "example_use": "Long-distance trip analysis.",
    },
    {
        "column_name": "is_long_duration_trip",
        "description": "Flag for trips above the 99th percentile of trip duration.",
        "data_type": "boolean",
        "example_use": "Long-duration trip analysis.",
    },
]

final_data_dictionary = pd.DataFrame(data_dictionary_entries)

final_data_dictionary_path = processed_data_dir / "final_tableau_data_dictionary.csv"
final_data_dictionary.to_csv(final_data_dictionary_path, index=False)

final_data_dictionary.head()

,column_name,description,data_type,example_use
0,pickup_datetime,Date and time when the taxi trip started.,datetime,Time trend analysis and filtering.
1,dropoff_datetime,Date and time when the taxi trip ended.,datetime,Trip duration calculation.
2,pickup_date,Date of taxi pickup.,date,Daily demand and revenue trend analysis.
3,pickup_hour,"Hour of day when the trip started, from 0 to 23.",integer,Peak demand hour analysis.
4,pickup_day_name,Day name of trip pickup.,categorical,Weekday demand comparison.


## Tableau File Documentation

Write the guide for Tableau inputs.

In [25]:
tableau_file_documentation = """# Tableau Data Files

Final datasets for Tableau dashboard development.

## Recommended Main File

### `tableau_taxi_trips_sample.csv`

Use this row-level sample for Tableau Public dashboards with filters, KPIs, maps, time views, payment analysis, and trip behavior analysis.

## Aggregated Summary Files

### `tableau_executive_kpi_summary.csv`

Used for executive KPI cards.

### `tableau_daily_summary.csv`

Used for daily demand and revenue trend charts.

### `tableau_hourly_summary.csv`

Used for peak-hour analysis.

### `tableau_weekday_summary.csv`

Used for weekday and weekend comparison.

### `tableau_pickup_borough_summary.csv`

Used for origin borough analysis.

### `tableau_dropoff_borough_summary.csv`

Used for destination borough analysis.

### `tableau_pickup_zone_summary.csv`

Used for detailed pickup zone analysis.

### `tableau_dropoff_zone_summary.csv`

Used for detailed destination zone analysis.

### `tableau_borough_flow_summary.csv`

Used for pickup borough to drop-off borough movement analysis.

### `tableau_zone_flow_summary.csv`

Used for pickup zone to drop-off zone flow analysis. Only flows with at least 50 trips are included.

### `tableau_payment_type_summary.csv`

Used for payment and tip behavior analysis.

### `tableau_distance_bucket_summary.csv`

Used for trip distance segment analysis.

### `tableau_duration_bucket_summary.csv`

Used for trip duration segment analysis.

### `tableau_hour_weekday_heatmap.csv`

Used for demand heatmaps by pickup hour and day of week.

### `tableau_outlier_flag_summary.csv`

Used for high-value, long-distance, and long-duration trip flag summaries.

## Instructor Preview File

### `instructor_preview_sample_10000.csv`

Small row-level sample for quick manual inspection.

## Notes

- Raw data remains unchanged in `data/raw/`.
- Tableau-ready files are stored in `data/processed/`.
- The main dashboard file is `tableau_taxi_trips_sample.csv`.
"""

with open(docs_dir / "tableau_data_files.md", "w", encoding="utf-8") as file:
    file.write(tableau_file_documentation)

print("Created docs/tableau_data_files.md")

Created docs/tableau_data_files.md


## Project Data Dictionary Markdown

Save the final field dictionary for documentation.

In [26]:
data_dictionary_markdown_rows = []

for _, row in final_data_dictionary.iterrows():
    data_dictionary_markdown_rows.append(
        f"| {row['column_name']} | {row['data_type']} | {row['description']} | {row['example_use']} |"
    )

data_dictionary_markdown = f"""# Data Dictionary

## Project

NYC Taxi Trip Analytics

## Main Tableau Dataset

`data/processed/tableau_taxi_trips_sample.csv`

## Field Descriptions

| Column Name | Data Type | Description | Example Use |
|---|---|---|---|
{chr(10).join(data_dictionary_markdown_rows)}

## Notes

The dataset was prepared from NYC Yellow Taxi Trip Records for January 2024. The raw file was not modified. Cleaning, feature engineering, and Tableau preparation were completed using Python notebooks.
"""

with open(docs_dir / "data_dictionary.md", "w", encoding="utf-8") as file:
    file.write(data_dictionary_markdown)

print("Updated docs/data_dictionary.md")

Updated docs/data_dictionary.md


## Dashboard Link Placeholder

Create the Tableau Public URL placeholder.

In [27]:
dashboard_links_path = tableau_dir / "dashboard_links.md"

if not dashboard_links_path.exists() or dashboard_links_path.read_text(encoding="utf-8").strip() == "":
    dashboard_links_content = """# Tableau Dashboard Links

## Final Dashboard

Tableau Public URL:

```text
To be added after publishing the dashboard.
```

## Notes

The final Tableau Public URL should be pasted above before final submission.
"""

    with open(dashboard_links_path, "w", encoding="utf-8") as file:
        file.write(dashboard_links_content)

    print("Created tableau/dashboard_links.md placeholder.")
else:
    print("tableau/dashboard_links.md already exists and was not overwritten.")

Created tableau/dashboard_links.md placeholder.


## Final Load Summary

Record the files created by this notebook.

In [28]:
final_load_summary_rows = []

for file_name, dataframe in output_files.items():
    final_load_summary_rows.append({
        "file_name": file_name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
    })

final_load_summary_rows.append({
    "file_name": "instructor_preview_sample_10000.csv",
    "rows": instructor_preview_dataset.shape[0],
    "columns": instructor_preview_dataset.shape[1],
})

final_load_summary_rows.append({
    "file_name": "final_tableau_data_dictionary.csv",
    "rows": final_data_dictionary.shape[0],
    "columns": final_data_dictionary.shape[1],
})

final_load_summary = pd.DataFrame(final_load_summary_rows)

final_load_summary_path = processed_data_dir / "final_load_summary.csv"
final_load_summary.to_csv(final_load_summary_path, index=False)

final_load_summary

,file_name,rows,columns
0,tableau_taxi_trips_sample.csv,300000,39
1,tableau_executive_kpi_summary.csv,13,3
2,tableau_daily_summary.csv,31,10
3,tableau_hourly_summary.csv,24,10
4,tableau_weekday_summary.csv,7,12
5,tableau_pickup_borough_summary.csv,7,12
6,tableau_dropoff_borough_summary.csv,7,12
7,tableau_pickup_zone_summary.csv,257,11
8,tableau_dropoff_zone_summary.csv,260,11
9,tableau_borough_flow_summary.csv,46,9


In [29]:
final_load_summary_markdown = f"""# Final Load Preparation Summary

## Purpose

This document summarizes the final datasets prepared for Tableau dashboard development.

## Source Dataset

- Cleaned input file: `data/processed/cleaned_yellow_taxi_jan_2024.parquet`
- Cleaned rows: {len(taxi_trips):,}
- Cleaned columns: {taxi_trips.shape[1]:,}

## Main Tableau File

- `data/processed/tableau_taxi_trips_sample.csv`
- Rows: {len(tableau_sample_dataset):,}
- Purpose: Main row-level file for Tableau Public dashboard development.

## Files Created

{final_load_summary.to_markdown(index=False)}

## Recommended Tableau Dashboard Sections

1. Executive KPI Overview
2. Demand by Hour and Day
3. Revenue and Fare Analysis
4. Pickup and Drop-off Borough Analysis
5. Zone-Level Demand Analysis
6. Payment and Tip Behavior
7. Trip Distance and Duration Efficiency
8. Business Recommendation View

## Notes

The final Tableau files are exported in CSV format for compatibility and easy inspection. Aggregated files are provided to improve Tableau performance and simplify dashboard building.
"""

with open(docs_dir / "final_load_summary.md", "w", encoding="utf-8") as file:
    file.write(final_load_summary_markdown)

print("Created docs/final_load_summary.md")

Created docs/final_load_summary.md


## Final Quality Check

Verify that expected output files exist.

In [30]:
expected_output_file_names = list(output_files.keys()) + [
    "instructor_preview_sample_10000.csv",
    "final_tableau_data_dictionary.csv",
    "final_load_summary.csv",
]

missing_output_files = []

for file_name in expected_output_file_names:
    file_path = processed_data_dir / file_name
    if not file_path.exists():
        missing_output_files.append(file_name)

if missing_output_files:
    raise FileNotFoundError(f"Missing output files: {missing_output_files}")

print("All expected final load files were created successfully.")
print(f"Number of final CSV outputs checked: {len(expected_output_file_names)}")

All expected final load files were created successfully.
Number of final CSV outputs checked: 19


# Final Load Preparation Summary

Notebook 05 created Tableau-ready CSV files and documentation.

## Main Outputs

- `data/processed/tableau_taxi_trips_sample.csv`
- `data/processed/tableau_executive_kpi_summary.csv`
- `data/processed/tableau_daily_summary.csv`
- `data/processed/tableau_hourly_summary.csv`
- `data/processed/tableau_weekday_summary.csv`
- `data/processed/tableau_pickup_borough_summary.csv`
- `data/processed/tableau_dropoff_borough_summary.csv`
- `data/processed/tableau_pickup_zone_summary.csv`
- `data/processed/tableau_dropoff_zone_summary.csv`
- `data/processed/tableau_borough_flow_summary.csv`
- `data/processed/tableau_zone_flow_summary.csv`
- `data/processed/tableau_payment_type_summary.csv`
- `data/processed/tableau_distance_bucket_summary.csv`
- `data/processed/tableau_duration_bucket_summary.csv`
- `data/processed/tableau_hour_weekday_heatmap.csv`
- `data/processed/tableau_outlier_flag_summary.csv`
- `data/processed/instructor_preview_sample_10000.csv`
- `data/processed/final_tableau_data_dictionary.csv`
- `data/processed/final_load_summary.csv`

## Documentation Outputs

- `docs/data_dictionary.md`
- `docs/tableau_data_files.md`
- `docs/final_load_summary.md`
- `tableau/dashboard_links.md`

## Next Step

Build the Tableau dashboard using the final CSV files from `data/processed/`.